# Web Scraping data
v1

In [1]:
import asyncio
from playwright.async_api import async_playwright
from playwright_stealth import Stealth
from bs4 import BeautifulSoup
import pandas as pd

In [7]:
async def scrape_bieber_full_info():
    async with Stealth().use_async(async_playwright()) as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        url = "https://www.prestigeonline.com/id/pursuits/entertainment/records-set-and-broken-by-justin-bieber-at-coachella-2026/"
        
        try:
            await page.goto(url, wait_until="networkidle", timeout=60000)
            await asyncio.sleep(5) 

            content = await page.content()
            soup = BeautifulSoup(content, "html.parser")

            # 1. PUBLICATION DATE (Specific to Prestige/News formats)
            # Checking multiple common patterns found in 2026 articles
            pub_date = "N/A"
            date_tag = soup.find(['time', 'span', 'div'], class_=['post-date', 'published', 'date', 'entry-date'])
            if date_tag:
                pub_date = date_tag.get_text(strip=True)

            # 2. TITLE
            title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "N/A"

            # 3. SUB-TITLE (The specific column you requested)
            # Often found in h2, or a paragraph with a 'lead' or 'excerpt' class
            subtitle = "N/A"
            subtitle_tag = soup.find(['h2', 'p'], class_=['excerpt', 'lead', 'article-subtitle', 'entry-summary'])
            if subtitle_tag:
                subtitle = subtitle_tag.get_text(strip=True)
            elif soup.find('h2'): # Fallback to the first H2 if no specific class exists
                subtitle = soup.find('h2').get_text(strip=True)

            # 4. CLEAN TEXT CONTENT
            # Decomposing non-content elements to avoid "cookie" and "IP" noise
            for junk in soup(["script", "style", "nav", "footer", "header", "aside", ".related", ".suggested", ".recommended", ".read-more", ".newsletter",".social-share", ".ads", ".sidebar", ".comment-area" ]):
                junk.decompose()
            
            for p in soup.find_all(['p','h3', 'div']):
                junk

                text = p.get_text().lower()
                if any (phrase in text for phrase in junk): 
                    if len(p.get_text()) < 200:
                        p.decompose()


            # Target the main article body specifically
            main_content = soup.find(['article', 'div'], class_=['entry-content', 'article-body', 'post-content'])
            clean_text = main_content.get_text(separator=' ', strip=True) if main_content else soup.get_text(separator=' ', strip=True)

            # 5. IMAGES
            images = [img.get('src') or img.get('data-src') for img in soup.find_all('img') 
                      if (img.get('src') or img.get('data-src')) and "http" in (img.get('src') or img.get('data-src'))]
            

            # 6. TAGS EXTRACTION (New Section)
            # Look for common tag containers and extract all links/text within them
            tags_list = []
            tag_container = soup.find(['div', 'span', 'ul'], class_=['tags', 'post-tags', 'entry-tags', 'topic-list'])
            if tag_container:
                # Find all links or spans inside that container
                tags_list = [t.get_text(strip=True) for t in tag_container.find_all(['a', 'span'])]

            # 7. EXPORT
            scraped_data = {
                "Publication Date": pub_date,
                "Title": title,
                "Sub-title": subtitle,
                "Clean Content": clean_text,
                "Image URLs": ", ".join(images[:10]),
                "Tags": ", ".join(set(tags_list)), # Using set() to remove duplicates
            }

            df = pd.DataFrame([scraped_data])
            df.to_csv("bieber_coachella_final.csv", index=False)
            print(f"Success! Data saved with Date: {pub_date} and Sub-title: {subtitle[:50]}...")

        except Exception as e:
            print(f"Error: {e}")
        finally:
            await browser.close()

await scrape_bieber_full_info()

Success! Data saved with Date: Published:May 02, 2026 06:08 AM WIT|5 MIN READ and Sub-title: Here are all the records set and broken by Justin ...


=====

In [1]:
import asyncio
import json
import random
from playwright.async_api import async_playwright
from playwright_stealth import Stealth
from bs4 import BeautifulSoup

In [2]:
async def scrape_multiple_to_single_json(url_list, output_file):
    all_articles = [] # This list will hold all our data
    
    async with Stealth().use_async(async_playwright()) as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        for url in url_list:
            print(f"Scraping: {url}")
            try:
                # 1. Navigation & Anti-Bot Wait
                await page.goto(url, wait_until="networkidle", timeout=60000)
                await asyncio.sleep(random.uniform(3, 7)) # Random wait to look human

                content = await page.content()
                soup = BeautifulSoup(content, "html.parser")

                # 2. Extract Metadata
                title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "N/A"
                
                subtitle = "N/A"
                sub_tag = soup.find(['h2', 'p'], class_=['excerpt', 'lead', 'article-subtitle'])
                if sub_tag: subtitle = sub_tag.get_text(strip=True)

                pub_date = "N/A"
                date_tag = soup.find(['time', 'span'], class_=['date', 'published', 'post-date'])
                if date_tag: pub_date = date_tag.get_text(strip=True)

                # 3. Extract Tags
                tags = [t.get_text(strip=True) for t in soup.select('.tags a, .post-tags span, .topic-list li')]

                # 4. Deep Clean Content (Remove suggestions/related stories)
                for junk in soup.select('script, style, nav, footer, header, aside, .related, .suggested, .read-more'):
                    junk.decompose()
                
                body_container = soup.find(['article', 'div'], class_=['entry-content', 'article-body'])
                clean_text = body_container.get_text(separator=' ', strip=True) if body_container else soup.get_text(separator=' ', strip=True)

                # 5. Create a Dictionary for this specific URL
                article_data = {
                    "url": url,
                    "title": title,
                    "subtitle": subtitle,
                    "date": pub_date,
                    "tags": list(set(tags)),
                    "content": clean_text
                }

                # 6. Add it to our master list
                all_articles.append(article_data)
                print(f"--- Saved: {title[:30]}...")

            except Exception as e:
                print(f"Error scraping {url}: {e}")
                continue

        await browser.close()

    # 7. Final Step: Save the entire list to 1 JSON file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_articles, f, ensure_ascii=False, indent=4)
    
    print(f"\nDONE! Total articles saved: {len(all_articles)} in {output_file}")

# --- LIST YOUR URLS HERE ---
urls_to_process = [
   "https://www.yahoo.com/entertainment/music/article/justin-bieber-is-headlining-coachella-this-month-how-the-pop-star-is-forging-a-new-path-163753630.html", 
"https://www.theguardian.com/music/2026/apr/10/coachella-2026-justin-bieber-sabrina-carpenter-karol-g", 
"https://www.rollingstone.com/music/music-news/justin-bieber-coachella-sneak-preview-1235540010/", 
"https://pagesix.com/2026/04/04/celebrity-news/justin-biebers-got-a-lot-riding-on-coachella-comeback/",
"https://eu.desertsun.com/story/entertainment/coachella/2026/04/03/justin-bieber-loves-palm-springs-but-it-doesnt-always-love-him-back-recap-of-infamous-moments/89319620007/",
"https://consequence.net/2026/03/justin-bieber-private-coachella-warmup-show-no-old-songs/",
"https://www.tmz.com/2026/04/08/justin-bieber-coachella-rehearsal-video/",
"https://www.thefader.com/2026/04/08/what-will-justin-bieber-play-at-coachella-2026",
"https://eu.desertsun.com/story/entertainment/coachella/2026/04/09/what-time-is-justin-bieber-performing-at-coachella/89452880007/",
"https://extratv.com/2026/04/10/justin-bieber-is-ready-for-big-comeback-with-coachella-headlining-performance/",
"https://www.yahoo.com/entertainment/music/articles/justin-bieber-shares-inside-look-223724367.html",
"https://www.tmz.com/2026/04/10/justin-bieber-fans-excited-after-hearing-coachella-rehearsal-videos/",
"https://www.yahoo.com/entertainment/music/articles/justin-bieber-leaked-coachella-rehearsals-172441820.html",
"https://ca.billboard.com/music/concerts/justin-bieber-private-show-details",
"https://www.laineygossip.com/justin-bieber-warms-up-for-coachella-with-surprise-25-song-set-at-the-roxy/",
"https://www.complex.com/music/a/tracewilliamcowen/justin-bieber-coachella-swag-album-guide",
"https://www.geo.tv/latest/658522-justin-bieber-coachella-performance-to-pave-way-for-greater-surprise",
"https://www.today.com/popculture/music/how-to-watch-coachella-free-at-home-rcna273650",
"https://eu.swtimes.com/story/shopping/entertainment/concert-tickets/2026/04/03/how-to-buy-coachella-2026-tickets-bieber-carpenter-karol-g/89451597007/",
"https://www.yahoo.com/entertainment/music/articles/inside-justin-bieber-preparations-headline-210251863.html",
"https://www.eonline.com/news/1430632/coachella-2026-kylie-jenner-alix-earle-celebrities",
"https://ca.billboard.com/music/music-news/justin-bieber-inside-pre-coachella-show-roxy-video-1236211833/",
"https://nypost.com/2026/04/08/us-news/what-to-pack-for-coachella-2026/",
"https://www.complex.com/music/a/alex-ocho/coachella-2026-live-streaming-guide-set-times-merchandise",
"https://www.movin925.com/watch-footage-of-justin-biebers-sneak-preview-of-coachella-at-las-the-roxy-theatre/,"
"https://eu.usatoday.com/videos/entertainment/music/2026/04/08/videos-show-justin-bieber-fans-prepare-for-coachella-bieberchella/89523384007/",
"https://www.vulture.com/article/coachella-2026-streaming-guide.html",
"https://parade.com/news/karol-g-set-to-become-first-latina-headline-coachella",
"https://www.teenvogue.com/story/how-to-livestream-coachella-2026",
"https://www.ctpublic.org/2026-04-10/coachella-2026-a-hand-picked-guide-to-the-best-of-the-fest", 
"https://eu.swtimes.com/videos/entertainment/music/2026/04/08/videos-show-justin-bieber-fans-prepare-for-coachella-bieberchella/89523384007/",
"https://www.aol.com/justin-bieber-headlining-coachella-month-163753408.html",
"https://www.yahoo.com/entertainment/articles/every-song-justin-bieber-played-164744463.html",
"https://www.elle.com/culture/celebrities/a70956825/kylie-jenner-photos-coachella-2026/",
"https://www.eonline.com/news/1430761/kylie-jenners-justin-bieber-coachella-inspired-nails",
"https://deadline.com/2026/04/coachella-2026-how-to-watch-livestreams-1236788018/",
"https://www.hits96.com/is-justin-bieber-only-going-to-play-new-songs-at-coachella/",
"https://www.newsweek.com/entertainment/justin-biebers-coachella-set-list-may-have-been-leaked-by-love-island-star-11808816",
"https://www.rollingstone.com/music/music-news/justin-bieber-first-us-concert-four-years-watch-videos-1235538722/",
"https://www.foxnews.com/entertainment/kylie-jenner-dances-face-mask-sports-bra-friends-coachella-weekend-kicks-desert",
"https://www.enidlive.com/music/hot-ac/watch-footage-of-justin-biebers-sneak-preview-of-coachella-at-las-the-roxy-theatre/",
"https://www.wionews.com/entertainment/hollywood/coachella-2026-justin-bieber-sabrina-carpenter-katseye-and-others-set-to-perform-in-california-1775827064067",
"https://eu.detroitnews.com/story/entertainment/music/2026/04/10/coachella-youtube-set-times-revealed-how-to-throw-your-own-couchellaeverything-to-know-for-couchella/89450951007/",
"https://eu.redding.com/videos/entertainment/music/2026/04/08/videos-show-justin-bieber-fans-prepare-for-coachella-bieberchella/89523384007/",
"https://www.geo.tv/latest/658675-justin-bieber-ignites-documentary-release-speculations-ahead-of-coachella",
"https://www.nbclosangeles.com/entertainment/2026-coachella-set-times/3872689/",
"https://www.fzine.com/culture/coachella-2026-lineup-schedule",
"https://eu.beaconjournal.com/story/entertainment/music/2026/04/10/watch-stream-sabrina-carpenter-justin-bieber-devo-coachella-2026-tickets/89549785007/",
"https://www.ocregister.com/2026/04/06/coachella-2026-set-times-schedule-for-sabrina-carpenter-justin-bieber-karol-g/",
"https://latination.com/coachella-2026-lineup-justin-bieber-sabrina-carpenter-united-states/",
"https://eu.azcentral.com/story/shopping/entertainment/concert-tickets/2026/04/03/how-to-buy-coachella-2026-tickets-bieber-carpenter-karol-g/89451597007/",
"https://www.tyla.com/entertainment/music/justin-bieber-coachella-setlist-067605-20260408",
"https://www.love105fm.com/2026/04/02/justin-biebers-roxy-takeover-a-sneak-preview-of-coachella-2026/",
"https://partners.zapzee.net/issue/article/112346/",
"https://patch.com/california/palmdesert/coachella-weekend-showtimes-announced-sabrina-carpenter-justin-bieber-karol-g",
"https://www.usmagazine.com/entertainment/news/inside-justin-biebers-coachella-2026-prep-setlist-filming/",
"https://ftw-eu.usatoday.com/story/entertainment/pop-culture/2026/04/09/coachella-2026-lineup-justin-bieber-sabrina-carpenter-performing/89519338007/",
"https://www.tag24.com/en/entertainment/celebrities/sabrina-carpenter/coachella-kicks-off-with-headliners-sabrina-carpenter-justin-bieber-and-karol-g-3488898",
"https://www.distractify.com/p/justin-bieber-2026-coachella-set-list-leak",
"https://www.billboard.com/pro/justin-bieber-coachella-radius-clause-roxy-show-how/",
"https://www.complex.com/music/a/cmplxtara-mahadevan/skylrk-new-collection-justin-bieber-coachella",
"https://www.tmz.com/2026/04/06/justin-bieber-performs-at-troubadour-before-coachella/",
"https://www.newsweek.com/entertainment/kylie-jenner-arrives-at-coachella-decked-out-in-justin-bieber-merch-11813350",
"https://www.rollingstone.com/music/music-lists/coachella-most-anticipated-acts-1235542311/"
    # Add your other news links here...
]


In [3]:
# Run the master function
await scrape_multiple_to_single_json(urls_to_process, "bieber_coachella_master.json")

Scraping: https://www.yahoo.com/entertainment/music/article/justin-bieber-is-headlining-coachella-this-month-how-the-pop-star-is-forging-a-new-path-163753630.html
--- Saved: guce...
Scraping: https://www.theguardian.com/music/2026/apr/10/coachella-2026-justin-bieber-sabrina-carpenter-karol-g
--- Saved: Coachella 2026: Justin Bieber ...
Scraping: https://www.rollingstone.com/music/music-news/justin-bieber-coachella-sneak-preview-1235540010/
--- Saved: N/A...
Scraping: https://pagesix.com/2026/04/04/celebrity-news/justin-biebers-got-a-lot-riding-on-coachella-comeback/
--- Saved: N/A...
Scraping: https://eu.desertsun.com/story/entertainment/coachella/2026/04/03/justin-bieber-loves-palm-springs-but-it-doesnt-always-love-him-back-recap-of-infamous-moments/89319620007/
--- Saved: Justin Bieber's best — and mos...
Scraping: https://consequence.net/2026/03/justin-bieber-private-coachella-warmup-show-no-old-songs/
Error scraping https://consequence.net/2026/03/justin-bieber-private-coachella-wa

===


In [1]:
import asyncio
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse
 
import httpx
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
URLS = [
    "https://www.yahoo.com/entertainment/music/article/justin-bieber-is-headlining-coachella-this-month-how-the-pop-star-is-forging-a-new-path-163753630.html",
    "https://www.theguardian.com/music/2026/apr/10/coachella-2026-justin-bieber-sabrina-carpenter-karol-g",
    "https://www.rollingstone.com/music/music-news/justin-bieber-coachella-sneak-preview-1235540010/",
    "https://pagesix.com/2026/04/04/celebrity-news/justin-biebers-got-a-lot-riding-on-coachella-comeback/",
    "https://eu.desertsun.com/story/entertainment/coachella/2026/04/03/justin-bieber-loves-palm-springs-but-it-doesnt-always-love-him-back-recap-of-infamous-moments/89319620007/",
    "https://consequence.net/2026/03/justin-bieber-private-coachella-warmup-show-no-old-songs/",
    "https://www.tmz.com/2026/04/08/justin-bieber-coachella-rehearsal-video/",
    "https://www.thefader.com/2026/04/08/what-will-justin-bieber-play-at-coachella-2026",
    "https://eu.desertsun.com/story/entertainment/coachella/2026/04/09/what-time-is-justin-bieber-performing-at-coachella/89452880007/",
    "https://extratv.com/2026/04/10/justin-bieber-is-ready-for-big-comeback-with-coachella-headlining-performance/",
    "https://www.yahoo.com/entertainment/music/articles/justin-bieber-shares-inside-look-223724367.html",
    "https://www.tmz.com/2026/04/10/justin-bieber-fans-excited-after-hearing-coachella-rehearsal-videos/",
    "https://www.yahoo.com/entertainment/music/articles/justin-bieber-leaked-coachella-rehearsals-172441820.html",
    "https://ca.billboard.com/music/concerts/justin-bieber-private-show-details",
    "https://www.laineygossip.com/justin-bieber-warms-up-for-coachella-with-surprise-25-song-set-at-the-roxy/",
    "https://www.complex.com/music/a/tracewilliamcowen/justin-bieber-coachella-swag-album-guide",
    "https://www.geo.tv/latest/658522-justin-bieber-coachella-performance-to-pave-way-for-greater-surprise",
    "https://www.today.com/popculture/music/how-to-watch-coachella-free-at-home-rcna273650",
    "https://eu.swtimes.com/story/shopping/entertainment/concert-tickets/2026/04/03/how-to-buy-coachella-2026-tickets-bieber-carpenter-karol-g/89451597007/",
    "https://www.yahoo.com/entertainment/music/articles/inside-justin-bieber-preparations-headline-210251863.html",
    "https://www.eonline.com/news/1430632/coachella-2026-kylie-jenner-alix-earle-celebrities",
    "https://ca.billboard.com/music/music-news/justin-bieber-inside-pre-coachella-show-roxy-video-1236211833/",
    "https://nypost.com/2026/04/08/us-news/what-to-pack-for-coachella-2026/",
    "https://www.complex.com/music/a/alex-ocho/coachella-2026-live-streaming-guide-set-times-merchandise",
    "https://www.movin925.com/watch-footage-of-justin-biebers-sneak-preview-of-coachella-at-las-the-roxy-theatre/",
    "https://eu.usatoday.com/videos/entertainment/music/2026/04/08/videos-show-justin-bieber-fans-prepare-for-coachella-bieberchella/89523384007/",
    "https://www.vulture.com/article/coachella-2026-streaming-guide.html",
    "https://parade.com/news/karol-g-set-to-become-first-latina-headline-coachella",
    "https://www.teenvogue.com/story/how-to-livestream-coachella-2026",
    "https://www.ctpublic.org/2026-04-10/coachella-2026-a-hand-picked-guide-to-the-best-of-the-fest",
    "https://eu.swtimes.com/videos/entertainment/music/2026/04/08/videos-show-justin-bieber-fans-prepare-for-coachella-bieberchella/89523384007/",
    "https://www.aol.com/justin-bieber-headlining-coachella-month-163753408.html",
    "https://www.yahoo.com/entertainment/articles/every-song-justin-bieber-played-164744463.html",
    "https://www.elle.com/culture/celebrities/a70956825/kylie-jenner-photos-coachella-2026/",
    "https://www.eonline.com/news/1430761/kylie-jenners-justin-bieber-coachella-inspired-nails",
    "https://deadline.com/2026/04/coachella-2026-how-to-watch-livestreams-1236788018/",
    "https://www.hits96.com/is-justin-bieber-only-going-to-play-new-songs-at-coachella/",
    "https://www.newsweek.com/entertainment/justin-biebers-coachella-set-list-may-have-been-leaked-by-love-island-star-11808816",
    "https://www.rollingstone.com/music/music-news/justin-bieber-first-us-concert-four-years-watch-videos-1235538722/",
    "https://www.foxnews.com/entertainment/kylie-jenner-dances-face-mask-sports-bra-friends-coachella-weekend-kicks-desert",
    "https://www.enidlive.com/music/hot-ac/watch-footage-of-justin-biebers-sneak-preview-of-coachella-at-las-the-roxy-theatre/",
    "https://www.wionews.com/entertainment/hollywood/coachella-2026-justin-bieber-sabrina-carpenter-katseye-and-others-set-to-perform-in-california-1775827064067",
    "https://eu.detroitnews.com/story/entertainment/music/2026/04/10/coachella-youtube-set-times-revealed-how-to-throw-your-own-couchellaeverything-to-know-for-couchella/89450951007/",
    "https://eu.redding.com/videos/entertainment/music/2026/04/08/videos-show-justin-bieber-fans-prepare-for-coachella-bieberchella/89523384007/",
    "https://www.geo.tv/latest/658675-justin-bieber-ignites-documentary-release-speculations-ahead-of-coachella",
    "https://www.nbclosangeles.com/entertainment/2026-coachella-set-times/3872689/",
    "https://www.fzine.com/culture/coachella-2026-lineup-schedule",
    "https://eu.beaconjournal.com/story/entertainment/music/2026/04/10/watch-stream-sabrina-carpenter-justin-bieber-devo-coachella-2026-tickets/89549785007/",
    "https://www.ocregister.com/2026/04/06/coachella-2026-set-times-schedule-for-sabrina-carpenter-justin-bieber-karol-g/",
    "https://latination.com/coachella-2026-lineup-justin-bieber-sabrina-carpenter-united-states/",
    "https://eu.azcentral.com/story/shopping/entertainment/concert-tickets/2026/04/03/how-to-buy-coachella-2026-tickets-bieber-carpenter-karol-g/89451597007/",
    "https://www.tyla.com/entertainment/music/justin-bieber-coachella-setlist-067605-20260408",
    "https://www.love105fm.com/2026/04/02/justin-biebers-roxy-takeover-a-sneak-preview-of-coachella-2026/",
    "https://partners.zapzee.net/issue/article/112346/",
    "https://patch.com/california/palmdesert/coachella-weekend-showtimes-announced-sabrina-carpenter-justin-bieber-karol-g",
    "https://www.usmagazine.com/entertainment/news/inside-justin-biebers-coachella-2026-prep-setlist-filming/",
    "https://ftw-eu.usatoday.com/story/entertainment/pop-culture/2026/04/09/coachella-2026-lineup-justin-bieber-sabrina-carpenter-performing/89519338007/",
    "https://www.tag24.com/en/entertainment/celebrities/sabrina-carpenter/coachella-kicks-off-with-headliners-sabrina-carpenter-justin-bieber-and-karol-g-3488898",
    "https://www.distractify.com/p/justin-bieber-2026-coachella-set-list-leak",
    "https://www.billboard.com/pro/justin-bieber-coachella-radius-clause-roxy-show-how/",
    "https://www.complex.com/music/a/cmplxtara-mahadevan/skylrk-new-collection-justin-bieber-coachella",
    "https://www.tmz.com/2026/04/06/justin-bieber-performs-at-troubadour-before-coachella/",
    "https://www.newsweek.com/entertainment/kylie-jenner-arrives-at-coachella-decked-out-in-justin-bieber-merch-11813350",
    "https://www.rollingstone.com/music/music-lists/coachella-most-anticipated-acts-1235542311/",
]

In [3]:
from pathlib import Path
OUTPUT_DIR = Path("output_2")
OUTPUT_DIR.mkdir(exist_ok=True)
 
JSONL_FILE = OUTPUT_DIR / "raw_articles.jsonl"
PARQUET_FILE = OUTPUT_DIR / "articles.parquet"
FAILED_FILE = OUTPUT_DIR / "failed_urls.txt"
CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint.json"

 
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}
 
CONCURRENCY = 8          # parallel async requests
DELAY_BETWEEN = 10      # seconds between requests (be polite)
PLAYWRIGHT_TIMEOUT = 25  # seconds
HTTP_TIMEOUT = 20
 
 

In [4]:

def load_checkpoint() -> set:
    if CHECKPOINT_FILE.exists():
        data = json.loads(CHECKPOINT_FILE.read_text())
        return set(data.get("done", []))
    return set()
 
def save_checkpoint(done: set):
    CHECKPOINT_FILE.write_text(json.dumps({"done": list(done)}))
 
def append_jsonl(record: dict):
    with JSONL_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
 
def is_allowed_by_robots(url: str) -> bool:
    try:
        parsed = urlparse(url)
        robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
        rp = RobotFileParser()
        rp.set_url(robots_url)
        rp.read()
        return rp.can_fetch("*", url)
    except Exception:
        return True  # if robots.txt unreachable, assume allowed
 
def parse_html(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")
 
    # Remove noise
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
 
    title = soup.title.string.strip() if soup.title else None
 
    # Try to find article body
    article = (
        soup.find("article")
        or soup.find(attrs={"class": re.compile(r"article|content|body|story", re.I)})
        or soup.find("main")
        or soup.body
    )
    text = article.get_text(separator=" ", strip=True) if article else ""
    text = re.sub(r"\s+", " ", text)  
 
    # Meta tags
    def meta(name):
        tag = soup.find("meta", attrs={"name": name}) or soup.find("meta", attrs={"property": name})
        return tag.get("content", "").strip() if tag else None
 
    return {
        "url": url,
        "domain": urlparse(url).netloc,
        "title": title,
        "description": meta("description") or meta("og:description"),
        "author": meta("author") or meta("article:author"),
        "published_date": meta("article:published_time") or meta("og:article:published_time"),
        "keywords": meta("keywords"),
        "text": text,
        "text_length": len(text),
        "scraped_at": datetime.now(timezone.utc).isoformat(),
        "method": None,   # filled by caller
        "status": None,   # filled by caller
        "error": None,
        "raw_html": html[:50000],
    }
 

In [5]:
# ─── Phase 1: Classification ───────────────────────────────────────────────────
 
async def classify_urls(urls: list[str]) -> dict:
    """Quick HEAD requests to classify each URL before full scraping."""
    results = {"ok": [], "blocked": [], "robots_blocked": [], "error": []}
 
    print("\n📋 Phase 1: Classifying URLs...")
 
    async with httpx.AsyncClient(headers=HEADERS, follow_redirects=True, timeout=10) as client:
        for url in urls:
            if not is_allowed_by_robots(url):
                results["robots_blocked"].append(url)
                print(f"  🤖 robots.txt blocked: {url}")
                continue
            try:
                r = await client.head(url)
                if r.status_code in (200, 301, 302):
                    results["ok"].append(url)
                    print(f"  ✅ {r.status_code} {url}")
                elif r.status_code in (403, 401, 429):
                    results["blocked"].append(url)
                    print(f"  🚫 {r.status_code} {url}")
                else:
                    results["error"].append(url)
                    print(f"  ⚠️  {r.status_code} {url}")
            except Exception as e:
                results["error"].append(url)
                print(f"  ❌ error: {url} — {e}")
            await asyncio.sleep(0.2)
 
    print(f"\n  Summary → OK: {len(results['ok'])} | Blocked: {len(results['blocked'])} | "
          f"robots.txt: {len(results['robots_blocked'])} | Error: {len(results['error'])}")
    return results
 

In [6]:
# Add this import (likely in the cell with other imports like asyncio, json, etc.)
import httpx

# Add this import (likely in the cell with other imports like asyncio, json, etc.)
import asyncio
import httpx

async def scrape_http(url: str, client: httpx.AsyncClient, semaphore: asyncio.Semaphore) -> dict:
    async with semaphore:
        try:
            r = await client.get(url, timeout=HTTP_TIMEOUT)
            record = parse_html(r.text, url)
            record["method"] = "httpx"
            record["status"] = r.status_code
            return record
        except Exception as e:
            return {
                "url": url,
                "domain": urlparse(url).netloc,
                "title": None, "description": None, "author": None,
                "published_date": None, "keywords": None, "text": None,
                "text_length": 0,
                "scraped_at": datetime.now(timezone.utc).isoformat(),
                "method": "httpx", "status": None, "error": str(e),
            }
 
async def run_http_phase(urls: list[str], done: set) -> tuple[list, list]:
    pending = [u for u in urls if u not in done]
    if not pending:
        print("\n⏭️  All HTTP URLs already done (checkpoint).")
        return [], []
 
    print(f"\n🌐 Phase 2: HTTP scraping {len(pending)} URLs...")
    semaphore = asyncio.Semaphore(CONCURRENCY)
    succeeded, failed = [], []
 
    async with httpx.AsyncClient(headers=HEADERS, follow_redirects=True) as client:
        tasks = [scrape_http(u, client, semaphore) for u in pending]
        for i, coro in enumerate(asyncio.as_completed(tasks)):
            record = await coro
            await asyncio.sleep(DELAY_BETWEEN)
 
            if record.get("error") or not record.get("text"):
                failed.append(record["url"])
                print(f"  ❌ [{i+1}/{len(pending)}] FAILED — {record['url']}")
            else:
                succeeded.append(record)
                done.add(record["url"])
                append_jsonl(record)
                save_checkpoint(done)
                print(f"  ✅ [{i+1}/{len(pending)}] {record['title'] or record['url'][:60]}")
 
    return succeeded, failed
 

In [7]:
# ─── Phase 3: Playwright for JS-heavy / failed sites ─────────────────────────
 
async def scrape_playwright(url: str, page) -> dict:
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=PLAYWRIGHT_TIMEOUT * 1000)
        await asyncio.sleep(2)  # let JS settle
        html = await page.content()
        record = parse_html(html, url)
        record["method"] = "playwright"
        record["status"] = 200
        return record
    except Exception as e:
        return {
            "url": url,
            "domain": urlparse(url).netloc,
            "title": None, "description": None, "author": None,
            "published_date": None, "keywords": None, "text": None,
            "text_length": 0,
            "scraped_at": datetime.now(timezone.utc).isoformat(),
            "method": "playwright", "status": None, "error": str(e),
        }
 
async def run_playwright_phase(urls: list[str], done: set) -> tuple[list, list]:
    pending = [u for u in urls if u not in done]
    if not pending:
        print("\n⏭️  No Playwright URLs to process.")
        return [], []
 
    try:
        from playwright.async_api import async_playwright
    except ImportError:
        print("\n⚠️  Playwright not installed. Run: pip install playwright && playwright install chromium")
        return [], pending
 
    print(f"\n🎭 Phase 3: Playwright scraping {len(pending)} URLs...")
    succeeded, failed = [], []
 
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent=HEADERS["User-Agent"],
            extra_http_headers={"Accept-Language": "en-US,en;q=0.5"}
        )
        page = await context.new_page()
 
        for i, url in enumerate(pending):
            record = await scrape_playwright(url, page)
            await asyncio.sleep(DELAY_BETWEEN)
 
            if record.get("error") or not record.get("text"):
                failed.append(url)
                print(f"  ❌ [{i+1}/{len(pending)}] FAILED — {url}")
            else:
                succeeded.append(record)
                done.add(url)
                append_jsonl(record)
                save_checkpoint(done)
                print(f"  ✅ [{i+1}/{len(pending)}] {record['title'] or url[:60]}")
 
        await browser.close()
 
    return succeeded, failed
 
 

In [8]:
# ─── Phase 4: Save to Parquet ─────────────────────────────────────────────────
 
def build_parquet():
    print("\n💾 Phase 4: Building Parquet file...")
    records = []
    if JSONL_FILE.exists():
        with JSONL_FILE.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
 
    if not records:
        print("  ⚠️  No records to save.")
        return
 
    df = pd.DataFrame(records)
 
    # Type cleanup for Parquet
    df["scraped_at"] = pd.to_datetime(df["scraped_at"], utc=True, errors="coerce")
    df["text_length"] = pd.to_numeric(df["text_length"], errors="coerce").fillna(0).astype(int)
    df["status"] = pd.to_numeric(df["status"], errors="coerce")
 
    df.to_parquet(PARQUET_FILE, index=False, compression="snappy", engine="pyarrow")
    print(f"  ✅ Saved {len(df)} rows → {PARQUET_FILE}")
    print(f"\n  Columns: {list(df.columns)}")
    print(f"  Successful: {df['error'].isna().sum()} | Failed: {df['error'].notna().sum()}")
    return df

In [9]:
# ─── Main ──────────────────────────────────────────────────────────────────────
 
from datetime import datetime
async def main():
    print("=" * 60)
    print("  Social Media News Scraper")
    print(f"  {len(URLS)} URLs | Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)
 
    done = load_checkpoint()
    if done:
        print(f"\n♻️  Resuming from checkpoint ({len(done)} URLs already done)")
 
    # Phase 1: Classify
    classification = await classify_urls(URLS)
 
    # URLs to try with HTTP first (ok + blocked — Playwright may bypass blocks)
    http_candidates = classification["ok"]
    playwright_candidates = classification["blocked"] + classification["error"]
    robots_blocked = classification["robots_blocked"]
 
    # Save robots-blocked list
    if robots_blocked:
        with FAILED_FILE.open("a") as f:
            for u in robots_blocked:
                f.write(f"ROBOTS_BLOCKED\t{u}\n")
 
    # Phase 2: HTTP
    _, http_failed = await run_http_phase(http_candidates, done)
 
    # Phase 3: Playwright (blocked + http-failed)
    playwright_targets = list(set(playwright_candidates + http_failed))
    _, pw_failed = await run_playwright_phase(playwright_targets, done)
 
    # Log final failures
    if pw_failed:
        with FAILED_FILE.open("a") as f:
            for u in pw_failed:
                f.write(f"FAILED\t{u}\n")
        print(f"\n⚠️  {len(pw_failed)} URLs could not be scraped — saved to {FAILED_FILE}")
 
    # Phase 4: Parquet
    df = build_parquet()
 
    print("\n" + "=" * 60)
    print("  ✅ Done!")
    print(f"  JSONL  → {JSONL_FILE}")
    print(f"  Parquet→ {PARQUET_FILE}")
    print(f"  Failed → {FAILED_FILE}")
    print("=" * 60)
 
 
if __name__ == "__main__":
    await main()
 

  Social Media News Scraper
  64 URLs | Started: 2026-05-06 21:05:41

📋 Phase 1: Classifying URLs...
  🚫 429 https://www.yahoo.com/entertainment/music/article/justin-bieber-is-headlining-coachella-this-month-how-the-pop-star-is-forging-a-new-path-163753630.html
  ✅ 200 https://www.theguardian.com/music/2026/apr/10/coachella-2026-justin-bieber-sabrina-carpenter-karol-g
  ✅ 200 https://www.rollingstone.com/music/music-news/justin-bieber-coachella-sneak-preview-1235540010/
  ✅ 200 https://pagesix.com/2026/04/04/celebrity-news/justin-biebers-got-a-lot-riding-on-coachella-comeback/
  ✅ 200 https://eu.desertsun.com/story/entertainment/coachella/2026/04/03/justin-bieber-loves-palm-springs-but-it-doesnt-always-love-him-back-recap-of-infamous-moments/89319620007/
  ✅ 200 https://consequence.net/2026/03/justin-bieber-private-coachella-warmup-show-no-old-songs/
  ✅ 200 https://www.tmz.com/2026/04/08/justin-bieber-coachella-rehearsal-video/
  ✅ 200 https://www.thefader.com/2026/04/08/what-will-jus